# Sentiment Timeline Analysis

This notebook is for analyzing the changes in student sentiment for course difficulty and quality over time, and seeing if these changes align with major LLM release dates.

The dataset used for this will be the rate-my-professors review dataset, scraped from rate my professors website

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next(path for path in [cwd, *cwd.parents] if (path / "src").exists())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.sentiment import (
    count_department_release_behaviors,
    load_releases,
    major_departments,
    overlay_releases,
    plot_department_ratings,
    plot_overall_ratings,
    plot_release_period_ratings,
    plot_smoothed_ratings,
    prepare_course_analysis,
    prepare_department_analysis,
    prepare_overall_analysis,
    prepare_review_dataset,
    summarize_department_ratings,
    summarize_department_release_behaviors,
    summarize_release_period_ratings,
)


In [ ]:
reviews_path = project_root / "data" / "rmp_ucsd_reviews.csv"
releases_path = project_root / "data" / "chatgpt_model_updates.csv"

analysis_start = pd.Timestamp("2016-01-01")
analysis_end = pd.Timestamp("2026-12-31")
classification_start = pd.Timestamp("2024-01-01")
classification_end = pd.Timestamp("2025-12-31")
selected_course = "PHYS2A"
course_list = ["MATH20C", "PHYS2A", "CHEM6A", "ECE35", "ECE45", "HUM3", "MMW11"]

reviews_df = prepare_review_dataset(reviews_path)
releases_df = load_releases(releases_path)


## Department Summary

In [ ]:
department_summary = summarize_department_ratings(reviews_df)
display(department_summary.round(3))


## Department Trends

In [ ]:
departments = major_departments(
    reviews_df,
    include=["ECE", "HUM", "CSE", "MATH"],
    top_n=8,
)

department_reviews, department_trends = prepare_department_analysis(
    df=reviews_df,
    departments=departments,
    start_date=analysis_start,
    end_date=analysis_end,
)

for department in departments:
    fig, ax = plot_department_ratings(department_trends, department)
    overlay_releases(ax, releases_df)
    plt.show()

for department in departments:
    department_release_summary = summarize_release_period_ratings(
        department_reviews.loc[department_reviews["department"] == department],
        releases_df=releases_df,
    )
    display(
        department_release_summary[[
            "release_period",
            "quality_mean",
            "difficulty_mean",
            "review_count",
        ]].round(3)
    )

    fig, ax = plot_release_period_ratings(
        department_release_summary,
        releases_df=releases_df,
        title=f"{department}: Average Ratings by LLM Release Period",
        start_date=analysis_start,
        end_date=analysis_end,
    )
    overlay_releases(ax, releases_df)
    plt.show()

selected_departments_release_summary = summarize_release_period_ratings(
    department_reviews,
    releases_df=releases_df,
)
display(
    selected_departments_release_summary[[
        "release_period",
        "quality_mean",
        "difficulty_mean",
        "review_count",
    ]].round(3)
)

fig, ax = plot_release_period_ratings(
    selected_departments_release_summary,
    releases_df=releases_df,
    title="Selected UCSD Departments Combined: Average Ratings by LLM Release Period",
    start_date=analysis_start,
    end_date=analysis_end,
)
overlay_releases(ax, releases_df)
plt.show()


## Department Release Behavior

In [ ]:
department_behavior_summary = summarize_department_release_behaviors(
    df=reviews_df,
    releases_df=releases_df,
    start_date=classification_start,
    end_date=classification_end,
    stable_threshold=0.15,
)
display(department_behavior_summary.round(3))

department_behavior_counts = count_department_release_behaviors(
    department_behavior_summary
)
display(department_behavior_counts)


## Overall Review Trends

In [ ]:
overall_reviews, overall_trends = prepare_overall_analysis(
    reviews_df,
    start_date=analysis_start,
    end_date=analysis_end,
)

fig, ax = plot_overall_ratings(
    overall_trends,
    title="UCSD Overall RMP Ratings Over Time (Smoothed)",
)
overlay_releases(ax, releases_df)
plt.show()

overall_release_summary = summarize_release_period_ratings(
    overall_reviews,
    releases_df=releases_df,
)
display(
    overall_release_summary[[
        "release_period",
        "quality_mean",
        "difficulty_mean",
        "review_count",
    ]].round(3)
)

fig, ax = plot_release_period_ratings(
    overall_release_summary,
    releases_df=releases_df,
    title="UCSD Overall: Average Ratings by LLM Release Period",
    start_date=analysis_start,
    end_date=analysis_end,
)
overlay_releases(ax, releases_df)
plt.show()


## Course Trends

In [ ]:
course_reviews, course_trends = prepare_course_analysis(
    reviews_df,
    course=selected_course,
    start_date=analysis_start,
    end_date=analysis_end,
)

fig, ax = plot_smoothed_ratings(course_trends, selected_course)
overlay_releases(ax, releases_df)
plt.show()

for course in course_list:
    _, smoothed_course = prepare_course_analysis(
        reviews_df,
        course=course,
        start_date=analysis_start,
        end_date=analysis_end,
    )
    fig, ax = plot_smoothed_ratings(smoothed_course, course)
    overlay_releases(ax, releases_df)
    plt.show()
